<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

In [1]:
#| echo: false
#| output: asis
show_doc(L1Loss)

---

[source](https://github.com/bmandracchia/bioMONAI/blob/main/bioMONAI/losses.py#L34){target="_blank" style="float:right; font-size:smaller"}

### L1Loss

>      L1Loss (inp:Any, targ:Any)

In [2]:
#| echo: false
#| output: asis
show_doc(MSELoss)

---

[source](https://github.com/bmandracchia/bioMONAI/blob/main/bioMONAI/losses.py#L27){target="_blank" style="float:right; font-size:smaller"}

### MSELoss

>      MSELoss (inp:Any, targ:Any)

In [3]:
#| echo: false
#| output: asis
show_doc(SSIMLoss)

---

### SSIMLoss

>      SSIMLoss (spatial_dims:int, data_range:float=1.0,
>                kernel_type:monai.metrics.regression.KernelType|str=gaussian,
>                win_size:int|collections.abc.Sequence[int]=11,
>                kernel_sigma:float|collections.abc.Sequence[float]=1.5,
>                k1:float=0.01, k2:float=0.03,
>                reduction:monai.utils.enums.LossReduction|str=mean)

Compute the loss function based on the Structural Similarity Index Measure (SSIM) Metric.

For more info, visit
    https://vicuesoft.com/glossary/term/ssim-ms-ssim/

SSIM reference paper:
    Wang, Zhou, et al. "Image quality assessment: from error visibility to structural
    similarity." IEEE transactions on image processing 13.4 (2004): 600-612.

### Combined Losses

In [4]:
#| echo: false
#| output: asis
show_doc(CombinedLoss)

---

[source](https://github.com/bmandracchia/bioMONAI/blob/main/bioMONAI/losses.py#L42){target="_blank" style="float:right; font-size:smaller"}

### CombinedLoss

>      CombinedLoss (spatial_dims=2, alpha=0.33, beta=0.33)

losses combined

In [5]:
#| echo: false
#| output: asis
show_doc(MSSSIMLoss)

---

[source](https://github.com/bmandracchia/bioMONAI/blob/main/bioMONAI/losses.py#L55){target="_blank" style="float:right; font-size:smaller"}

### MSSSIMLoss

>      MSSSIMLoss (spatial_dims=2, window_size:int=11, sigma:float=1.5,
>                  reduction:str='mean', levels:int=5, weights=None)

Base class for all neural network modules.

Your models should also subclass this class.

Modules can also contain other Modules, allowing to nest them in
a tree structure. You can assign the submodules as regular attributes::

    import torch.nn as nn
    import torch.nn.functional as F

    class Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(1, 20, 5)
            self.conv2 = nn.Conv2d(20, 20, 5)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            return F.relu(self.conv2(x))

Submodules assigned in this way will be registered, and will have their
parameters converted too when you call :meth:`to`, etc.

.. note::
    As per the example above, an ``__init__()`` call to the parent class
    must be made before assignment on the child.

:ivar training: Boolean represents whether this module is in training or
                evaluation mode.
:vartype training: bool

In [ ]:
msssim_loss = MSSSIMLoss(levels=3)
ssim_loss = SSIMLoss(2)
output = torch.rand(10, 3, 64, 64).cuda()  # Example output
target = torch.rand(10, 3, 64, 64).cuda()  # Example target
loss = msssim_loss(output, target)
loss2 = ssim_loss(output,target)
print("ms-ssim: ",loss, '\nssim: ', loss2)

ms-ssim:  tensor(0.9393, device='cuda:0') 
ssim:  tensor(0.9873, device='cuda:0')


In [ ]:
# #| export
# class MSSSIML1Loss(torch.nn.Module):
#     def __init__(self, alpha: float = 0.025, window_size: int = 11, sigma: float = 1.5, reduction: str = "mean", levels: int = 3, weights=None):
#         """
#         Multi-Scale Structural Similarity (MSSSIM) with L1 Loss.

#         Args:
#             alpha (float): Weighting factor between MS-SSIM and L1 loss. Controls the balance between the two losses.
#             window_size (int): Size of the Gaussian window for SSIM.
#             sigma (float): Standard deviation of the Gaussian.
#             reduction (str): Specifies the reduction to apply to the output ('mean', 'sum', or 'none').
#             levels (int): Number of scales to use for MS-SSIM.
#             weights (list): Weights to apply to each scale. If None, default values are used.
#         """
#         super(MSSSIML1Loss, self).__init__()
#         self.msssim = MSSSIMLoss(window_size=window_size, sigma=sigma, reduction="none", levels=levels, weights=weights)
#         self.alpha = alpha
#         self.reduction = reduction

#     def forward(self, x, y):
#         # Compute MSSSIM loss
#         msssim_loss = self.msssim(x, y)
        
#         # Compute L1 loss
#         l1_loss = F.l1_loss(x, y, reduction='none')
#         gaussian_weight = self.get_gaussian_weight(x.size())
#         weighted_l1_loss = (l1_loss * gaussian_weight).mean() / 2.0  # Weighted L1 loss

#         # Combine the two losses
#         combined_loss = self.alpha * (1 - msssim_loss) + (1 - self.alpha) * l1_loss

#         # Apply reduction (mean or sum)
#         if self.reduction == "mean":
#             return combined_loss.mean()
#         elif self.reduction == "sum":
#             return combined_loss.sum()
#         else:
#             return combined_loss
        
#     def get_gaussian_weight(self, size):
#         """Generate a Gaussian weight tensor based on input size."""
#         batch_size, channels, width, height = size
#         sigma = width / 6.0  # Using width/6 as an approximate scale for sigma

#         x = torch.arange(width, dtype=torch.float32, device='cuda')
#         y = torch.arange(height, dtype=torch.float32, device='cuda')

#         # Handle even-sized patches by adjusting the center position calculation
#         center_x = (width - 1) / 2.0 if width % 2 == 1 else width / 2.0
#         center_y = (height - 1) / 2.0 if height % 2 == 1 else height / 2.0

#         # Explicitly pass the indexing argument
#         x_grid, y_grid = torch.meshgrid(x, y, indexing='ij')

#         gaussian = torch.exp(-((x_grid - center_x)**2 + (y_grid - center_y)**2) / (2 * sigma**2))
#         gaussian /= gaussian.sum()  # Normalize the Gaussian

#         gaussian_weight = gaussian.view(1, 1, width, height)
#         gaussian_weight = gaussian_weight.expand(batch_size, channels, -1, -1)
#         return gaussian_weight

In [6]:
#| echo: false
#| output: asis
show_doc(MSSSIML1Loss)

---

[source](https://github.com/bmandracchia/bioMONAI/blob/main/bioMONAI/losses.py#L111){target="_blank" style="float:right; font-size:smaller"}

### MSSSIML1Loss

>      MSSSIML1Loss (spatial_dims=2, alpha:float=0.025, window_size:int=11,
>                    sigma:float=1.5, reduction:str='mean', levels:int=3,
>                    weights=None)

Base class for all neural network modules.

Your models should also subclass this class.

Modules can also contain other Modules, allowing to nest them in
a tree structure. You can assign the submodules as regular attributes::

    import torch.nn as nn
    import torch.nn.functional as F

    class Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(1, 20, 5)
            self.conv2 = nn.Conv2d(20, 20, 5)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            return F.relu(self.conv2(x))

Submodules assigned in this way will be registered, and will have their
parameters converted too when you call :meth:`to`, etc.

.. note::
    As per the example above, an ``__init__()`` call to the parent class
    must be made before assignment on the child.

:ivar training: Boolean represents whether this module is in training or
                evaluation mode.
:vartype training: bool

In [ ]:
msssiml1_loss = MSSSIML1Loss(alpha=0.025, window_size=11, sigma=1.5, levels=3)
input_image = torch.randn(4, 1, 128, 128)  # Batch of 4 grayscale images (1 channel)
target_image = torch.randn(4, 1, 128, 128)

# Compute MSSSIM + Gaussian-weighted L1 loss
loss = msssiml1_loss(input_image, target_image)
loss2 = ssim_loss(input_image, target_image)
print("ms-ssim: ", loss, '\nssim: ', loss2)

ms-ssim:  tensor(0.0249) 
ssim:  tensor(0.9958)


In [7]:
#| echo: false
#| output: asis
show_doc(MSSSIML2Loss)

---

[source](https://github.com/bmandracchia/bioMONAI/blob/main/bioMONAI/losses.py#L186){target="_blank" style="float:right; font-size:smaller"}

### MSSSIML2Loss

>      MSSSIML2Loss (spatial_dims=2, alpha:float=0.1, window_size:int=11,
>                    sigma:float=1.5, reduction:str='mean', levels:int=3,
>                    weights=None)

Base class for all neural network modules.

Your models should also subclass this class.

Modules can also contain other Modules, allowing to nest them in
a tree structure. You can assign the submodules as regular attributes::

    import torch.nn as nn
    import torch.nn.functional as F

    class Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(1, 20, 5)
            self.conv2 = nn.Conv2d(20, 20, 5)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            return F.relu(self.conv2(x))

Submodules assigned in this way will be registered, and will have their
parameters converted too when you call :meth:`to`, etc.

.. note::
    As per the example above, an ``__init__()`` call to the parent class
    must be made before assignment on the child.

:ivar training: Boolean represents whether this module is in training or
                evaluation mode.
:vartype training: bool

In [ ]:
msssim_l2_loss = MSSSIML2Loss()
output = torch.rand(10, 3, 64, 64).cuda()  # Example output with even dimensions
target = torch.rand(10, 3, 64, 64).cuda()  # Example target with even dimensions
loss = msssim_l2_loss(output, target)
print(loss)

tensor(0.0963, device='cuda:0')


### Dice Loss

In [8]:
#| echo: false
#| output: asis
show_doc(DiceLoss)

---

[source](https://github.com/bmandracchia/bioMONAI/blob/main/bioMONAI/losses.py#L261){target="_blank" style="float:right; font-size:smaller"}

### DiceLoss

>      DiceLoss (smooth=1)

DiceLoss computes the Sørensen–Dice coefficient loss, which is often used 
for evaluating the performance of image segmentation algorithms.

The Dice coefficient is a measure of overlap between two samples. It ranges 
from 0 (no overlap) to 1 (perfect overlap). The Dice loss is computed as 
1 - Dice coefficient, so it ranges from 1 (no overlap) to 0 (perfect overlap).

Attributes:
    smooth (float): A smoothing factor to avoid division by zero and ensure numerical stability.

Methods:
    forward(inputs, targets):
        Computes the Dice loss between the predicted probabilities (inputs) 
        and the ground truth (targets).

In [ ]:
# inputs and targets must be equally dimensional tensors
from torch import randn, randint

Dice Loss: 0.4994280934333801


In [ ]:
inputs = randn((1, 1, 256, 256))  # Input
targets = randint(0, 2, (1, 1, 256, 256)).float()  # Ground Truth

# Initialize
dice_loss = DiceLoss()

# Compute loss
loss = dice_loss(inputs, targets)
print('Dice Loss:', loss.item())

### Fourier Ring Correlation

In [9]:
#| echo: false
#| output: asis
show_doc(FRCLoss)

---

[source](https://github.com/bmandracchia/bioMONAI/blob/main/bioMONAI/losses.py#L314){target="_blank" style="float:right; font-size:smaller"}

### FRCLoss

>      FRCLoss (image1, image2)

Compute the Fourier Ring Correlation (FRC) loss between two images.

#### Args:
    - image1 (torch.Tensor): The first input image.
    - image2 (torch.Tensor): The second input image.

#### Returns:
    - torch.Tensor: The FRC loss.

In [10]:
#| echo: false
#| output: asis
show_doc(seventh_fourier_ring_correlation)

---

[source](https://github.com/bmandracchia/bioMONAI/blob/main/bioMONAI/losses.py#L331){target="_blank" style="float:right; font-size:smaller"}

### seventh_fourier_ring_correlation

>      seventh_fourier_ring_correlation (image1, image2)

Calculate the cutoff frequency at when Fourier ring correlation drops to 1/7.

#### Args:
    - image1 (torch.Tensor): The first input image.
    - image2 (torch.Tensor): The second input image.

#### Returns:
    - float: The cutoff frequency.

---